# Human-in-the-Loop: Gerbang Pra-Aksi, Peringkat Risiko, dan Pencatatan Audit

README untuk pelajaran ini memperkenalkan Human-in-the-Loop dengan potongan singkat yang meminta pengguna untuk `SETUJU` atau `TOLAK` setelah agen sudah menghasilkan respons. Pola itu adalah titik awal yang baik, tetapi implementasi HITL produksi biasanya memerlukan tiga hal tambahan:

1. Sebuah **gerbang pra-aksi** yang berjalan **sebelum** agen mengeksekusi langkah berisiko, sehingga biaya, ketidakbisaubahan, dan latensi tetap terjaga.
2. **Peringkat risiko**, sehingga aksi berisiko rendah dieksekusi otomatis, aksi berisiko sedang disetujui secara batch, dan hanya aksi berisiko tinggi yang memerlukan keterlibatan manusia.
3. Sebuah **log audit plus siklus revisi**, sehingga setiap keputusan gerbang dicatat sebagai JSONL, dan penolakan mengulang permintaan kepada agen dengan alasan terstruktur alih-alih hanya mencetak `Sedang merevisi...`.

Notebook ini membangun masing-masing dari hal tersebut di atas primitif yang sama seperti `06-system-message-framework.ipynb`. Notebook ini berjalan end-to-end dalam mode `DEMO_MODE = True` (tanpa input interaktif) atau dengan prompt `input()` nyata saat `DEMO_MODE = False`. Catatan: dalam mode DEMO, pengulangan tujuan ketiga diskripkan sehingga mekanisme pengulangan terlihat secara menyeluruh. Klasifikasi ulang yang dipandu oleh revisi sebenar membutuhkan `DEMO_MODE = False` dan operator.

**Di luar cakupan (ditangani di pelajaran lain):** autentikasi dan kontrol akses (Pelajaran 06 README ancaman 2), middleware pemanggilan alat (Pelajaran 14 MAF penjelasan mendalam), pola debat multi-agen.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pola 1: Gerbang pra-tindakan

Cuplikan HITL pada README memanggil agen terlebih dahulu, lalu meminta pengguna untuk menyetujui keluaran. Itu adalah alur **pasca-tindakan**. Agen sudah menjalankan proses, jadi biaya panggilan LLM sudah dibayar, dan efek samping apa pun (email yang dikirim, baris basis data yang ditulis, komentar yang diposting) sudah terjadi.

Alur **pra-tindakan** memasang gerbang sebelum agen menjalankan langkah berisiko. Agen mengusulkan tindakan, gerbang memutuskan apakah akan menjalankan, dan hanya dengan persetujuan efek samping tersebut terjadi.

| Aspek | Persetujuan pasca-tindakan (cuplikan README) | Gerbang pra-tindakan (notebook ini) |
|---|---|---|
| Kapan persetujuan dijalankan? | Setelah agen menghasilkan keluaran | Sebelum efek samping apapun dijalankan |
| Biaya LLM saat penolakan | Sudah dibayar | Hanya dibayar untuk usulan, bukan tindakan |
| Efek samping yang tidak dapat dibalik | Mungkin (tindakan sudah terjadi) | Dicegah |
| Kejelasan audit | Persetujuan adalah pernyataan cetak | Persetujuan adalah catatan JSON dengan cap waktu, tindakan, alasan |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pola 2: Pengelompokan Risiko

Tidak setiap tindakan membutuhkan persetujuan manusia. Pencarian hanya-baca terhadap API publik memiliki risiko yang berbeda dibandingkan dengan mengirim email kepada pelanggan. Memperlakukan keduanya sama hanya membuang perhatian operator dan memperlambat agen.

Model sederhana 3 tingkat:

| Tingkat | Contoh | Alur persetujuan |
|---|---|---|
| `rendah` (hanya-baca) | Mencari basis pengetahuan, mencari opsi penerbangan, mengambil halaman web publik | Eksekusi otomatis, dicatat untuk audit |
| `sedang` (mutasi murah) | Menyimpan hasil cache, menambah penghitung, menjadwalkan pengingat | Eksekusi otomatis, tetapi ditinjau harian secara bertahap |
| `tinggi` (menghadap eksternal atau tidak dapat dibatalkan) | Mengirim email, mengenakan biaya kartu, memposting ke saluran publik | Diblokir sampai persetujuan manusia |

Ini adalah satu pengelompokan. Sistem produksi sering menggunakan tingkat yang lebih rinci (misalnya, tingkat izin AWS IAM, tingkat akses berbasis peran). Versi 3 tingkat di bawah ini adalah versi terkecil yang berguna untuk agen yang menggabungkan tindakan hanya-baca dan berdampak samping.

Klasifikator di bawah ini menggunakan heuristik kata kunci supaya demo tetap deterministik dan murah. Dalam sistem produksi Anda akan mengganti ini dengan klasifikator yang dipelajari atau mesin kebijakan.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pola 3: Log audit dan siklus revisi

`print("Response approved.")` bukanlah sebuah log audit. Untuk membangun kepercayaan, setiap keputusan gerbang harus direkam sebagai sebuah peristiwa terstruktur yang nanti bisa Anda cari, putar ulang, atau lampirkan pada tinjauan insiden.

Dua bagian:

1. **JSONL hanya penambahan.** Satu baris per keputusan, dengan cap waktu, aksi, tingkat, keputusan, alasan. Mudah untuk dicari dengan grep, mudah untuk dikirim ke penyimpanan log nyata nanti.
2. **Siklus revisi saat penolakan.** Ketika gerbang mengembalikan `deny`, agen memberikan prompt ulang pada dirinya sendiri dengan alasan penolakan dalam konteks, sehingga proposal berikutnya bisa menghindari masalah tersebut.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Sumber daya tambahan

Beberapa proyek publik lain mengimplementasikan variasi pola HITL ini. Bandingkan pendekatan untuk menemukan yang sesuai dengan stack Anda:

- **LangChain** pembungkus alat human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): pembungkus alat siap pakai yang menghentikan eksekusi untuk masukan manusia.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ merestrukturisasi ini): menggunakan peran agen secara khusus untuk mewakili manusia dalam percakapan multi-agen.
- **Semantic Kernel** filter fungsi ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware yang berjalan di sekitar setiap panggilan fungsi, cocok untuk logika pengaturan akses.

Setiap proyek menangani tiga sub-pola tersebut secara berbeda: LangChain membungkusnya sebagai alat, AutoGen menggunakan peran agen, Semantic Kernel menggunakan filter middleware. Bacalah satu atau dua implementasi secara menyeluruh sebelum memilih desain untuk agen Anda sendiri.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
